# 04 · DINOv3 ViT-S/16 ReID encoder (final system)

Fine-tunes the crop encoder of the final system, *Swin + DINO* in `main.ipynb` (diagram 4). Each person crop (256×128) goes through a **DINOv3 ViT-S/16** pretrained with self-supervision. The embedding concatenates the CLS token with the mean of the upper and lower halves of the 16×8 patch grid, then a linear layer projects it to 256-d and L2-normalises it.

- **Trainable:** the last four transformer blocks and the final norm (lr 1e-5), plus the projection (lr 1e-4). AdamW with weight decay 1e-4.
- **Loss:** supervised NT-Xent (SupCon, τ = 0.1) on ground-truth crops, with random P×K = 16×4 identity batches and positives from different frames. No hard mining (`mining_config.enabled = False`).
- **Selection:** retrieval mAP on the validation frames after every epoch, over 120 epochs. The best epoch is published as `swin_dino_reid.pt`, and it was epoch 120.
- **Validation detector:** detections on the validation frames come from the frozen joint Swin-T model of `01_joint_person_search` (`joint_swin_t_triplet.pt`, 256×384). This is how the published model was selected. In `main.ipynb` the encoder is paired with the fine-tuned 512×768 detector (`swin_detector.pt`) instead.

**Run time (Kaggle T4):** W&B recorded ≈ 2.7 h for the published run.

### How to run
1. Create a Kaggle notebook from this file (*File → Import Notebook*), then attach the inputs listed below with *Add Input*.
2. Settings: **Accelerator → GPU T4 ×1**. Internet is only needed if you set `USE_WANDB = True`.
3. **Run All.** The inputs are found automatically wherever Kaggle mounts them under `/kaggle/input`.

Weights & Biases logging is **off** by default (`USE_WANDB = False`), so no account or secret is needed. To log curves and upload checkpoints, set `USE_WANDB = True` and store your key as the Kaggle secret / environment variable `WANDB_API_KEY`.

**Inputs**

| Input | Kaggle source | Used for |
|---|---|---|
| PRW dataset | `edoardomerli/prw-person-re-identification-in-the-wild` | frames, annotations, queries |
| Code | `saadqasimabbasi/prw-person-search-code` | the project modules in `src/` (and the DINOv3 code snapshot) |
| Validation detector | output of `01_joint_person_search` (`joint_swin_t_triplet.pt`) | detections on the validation frames |
| DINOv3 weights | a dataset containing `dinov3_vits16_pretrain_lvd1689m-08c60483.pth` | pretrained ViT-S/16 (from Meta's DINOv3 release, license required) |

The DINOv3 *architecture* code comes from the snapshot in the code dataset, so no Internet is needed. The pretrained *weights* are distributed by Meta under the DINOv3 license. Download them from the [DINOv3 repository](https://github.com/facebookresearch/dinov3) and upload them as a private Kaggle dataset.

## 1. Setup: locate the Kaggle inputs
This cell scans `/kaggle/input` once and finds the project code (the `prw-person-search-code` dataset), the PRW dataset and the checkpoint files this notebook needs. It fails with a clear message if one is missing, then adds the code folder to `sys.path`. Outputs go to `/kaggle/working`. Set the environment variables `PRW_INPUT_ROOT` / `PRW_WORKING` to run outside Kaggle.

> **Run time:** a few seconds.

In [ ]:
import os
import sys
from pathlib import Path

import torch

INPUT_ROOT = Path(os.environ.get("PRW_INPUT_ROOT", "/kaggle/input"))
WORKING = Path(os.environ.get("PRW_WORKING", "/kaggle/working"))
USE_WANDB = False  # True: log curves and upload checkpoints to W&B (needs WANDB_API_KEY)
os.environ["WANDB_MODE"] = "online" if USE_WANDB else "disabled"
os.environ["WANDB_DIR"] = str(WORKING)

PROJECT_MODULES = {"model.py", "training.py", "prw_dataset.py", "prw_evaluation.py"}
REQUIRED_FILES = {"joint_swin_t_triplet.pt", "dinov3_vits16_pretrain_lvd1689m-08c60483.pth"}
SKIPPED_FOLDERS = {"frames", "annotations", "query_box", ".git"}  # thousands of images, nothing to find


def scan_inputs():
    """Walk the input folder once and return the code, DINOv3, PRW and checkpoint locations."""
    code_dir = dinov3_dir = prw_root = None
    found = {}
    for directory, subfolders, files in os.walk(INPUT_ROOT, followlinks=True):
        directory = Path(directory)
        if prw_root is None and "query_info.txt" in files and "frames" in subfolders:
            prw_root = directory
        subfolders[:] = [name for name in subfolders if name not in SKIPPED_FOLDERS]
        if code_dir is None and PROJECT_MODULES.issubset(files):
            code_dir = directory
        if dinov3_dir is None and "hubconf.py" in files and directory.name.startswith("dinov3"):
            dinov3_dir = directory
        for name in REQUIRED_FILES.intersection(files):
            found.setdefault(name, directory / name)
    return code_dir, dinov3_dir, prw_root, found


CODE_DIR, DINOV3_DIR, PRW_ROOT, INPUT_FILES = scan_inputs()
missing = [label for label, value in [("code dataset (project .py files)", CODE_DIR),
                                      ("PRW dataset", PRW_ROOT)] if value is None]
missing += [f"input file {name}" for name in sorted(REQUIRED_FILES - set(INPUT_FILES))]
if DINOV3_DIR is None:
    missing.append("DINOv3 code snapshot (dinov3/hubconf.py in the code dataset)")
if missing:
    raise FileNotFoundError("Missing inputs (see 'How to run'): " + ", ".join(missing))
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
WORKING.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Code:  ", CODE_DIR)
print("PRW:   ", PRW_ROOT)
for name, path in sorted(INPUT_FILES.items()):
    print("Input: ", name, "->", path)
print("Device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "(no GPU: enable one in the settings)")

## 2. Frozen validation detector
The joint Swin-T model is loaded strictly and frozen. Its split seed defines the validation frames, so the encoder never trains on them. The checks make sure it is the expected 256×384 Swin-T/triplet checkpoint.

> **Run time:** ≈ 10 s.

In [ ]:
import random

from dino_reid import DinoPersonSearch, crop_epoch
from dinov3_reid import DinoV3CropReID
from model import PersonSearchModel

DETECTOR_CHECKPOINT = INPUT_FILES["joint_swin_t_triplet.pt"]
detector_checkpoint = torch.load(DETECTOR_CHECKPOINT, map_location="cpu", weights_only=False)
detector_config = dict(detector_checkpoint["model_config"])
if detector_config.get("backbone_name") != "swin_t" or detector_config.get("metric_loss") != "triplet":
    raise ValueError("Expected the Swin-T + triplet checkpoint from 01_joint_person_search")
if tuple(detector_checkpoint["run_config"]["image_size"]) != (256, 384):
    raise ValueError("This notebook assumes the 256x384 scene size of the joint model")
detector_config["pretrained"] = False
detector = PersonSearchModel(**detector_config).to(device)
detector.load_state_dict(detector_checkpoint["model_state_dict"], strict=True)
detector.requires_grad_(False).eval()
DETECTOR_EPOCH = detector_checkpoint["epoch"]
SEED = detector_checkpoint["run_config"]["seed"]
del detector_checkpoint
torch.manual_seed(SEED)
random.seed(SEED)
print("validation detector: epoch", DETECTOR_EPOCH, "seed", SEED)

## 3. Data: split and P×K crop batches
Same 90/10 frame split as the other notebooks. Only the ground-truth person crops (256×128) are used for training. `MinedPKIdentityBatchSampler` is the P×K sampler used in the published run. With no mined neighbours it draws random identity groups.

> **Run time:** ≈ 10 s.

In [ ]:
from torch.utils.data import DataLoader

from offline_mining import MinedPKIdentityBatchSampler
from prw_dataset import PRWPersonSearchDataset, PRWSubset, PKIdentityBatchSampler, prw_collate_fn

IMAGE_SIZE = (256, 384)
REID_CROP_SIZE = (256, 128)
VAL_FRACTION = 0.10
P, K = 16, 4
VAL_P, VAL_K = 4, 2
NUM_WORKERS = 2
HARD_FRACTION = .5  # only used when mined neighbours are set (never, here)

full_train = PRWPersonSearchDataset(PRW_ROOT, split="train", image_size=IMAGE_SIZE, query_crop_size=REID_CROP_SIZE)
indices = torch.randperm(len(full_train), generator=torch.Generator().manual_seed(SEED)).tolist()
val_count = round(len(indices) * VAL_FRACTION)
val_dataset = PRWSubset(full_train, indices[:val_count])
train_dataset = PRWSubset(full_train, indices[val_count:])
train_sampler = MinedPKIdentityBatchSampler(train_dataset, p=P, k=K, seed=SEED, hard_fraction=HARD_FRACTION)
val_sampler = PKIdentityBatchSampler(val_dataset, p=VAL_P, k=VAL_K, seed=SEED)
train_loader = DataLoader(train_dataset, batch_sampler=train_sampler, collate_fn=prw_collate_fn,
                          num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0)
val_loader = DataLoader(val_dataset, batch_sampler=val_sampler, collate_fn=prw_collate_fn,
                        num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0)
print(f"train frames: {len(train_dataset)}, validation frames: {len(val_dataset)}")

## 4. Fine-tune DINOv3 and the projection
`DinoV3CropReID` (`src/dinov3_reid.py`) builds ViT-S/16 from the local DINOv3 code and loads the pretrained weights. It unfreezes the last four blocks and adds the CLS + upper/lower pooling and projection. Each epoch:
1. trains on the P×K crop batches (`crop_epoch` in `src/dino_reid.py`);
2. computes the validation loss on fixed batches;
3. runs the retrieval validation: detect people on the validation frames with the frozen detector, embed the crops, rank against the validation queries.

`best.pt` keeps the highest validation mAP. Checkpoints include the RNG state, so `RESUME_FROM` can continue an interrupted run. The best model is copied to `/kaggle/working/swin_dino_reid.pt`.

> **Run time (Kaggle T4):** ≈ 1.3 min per epoch, ≈ 2.7 h for 120 epochs.

In [ ]:
import shutil

import wandb

from training import make_grad_scaler, start_run
from validation_retrieval import build_validation_gallery, evaluate_validation_gallery

NUM_EPOCHS = 120
VAL_MAX_BATCHES = 20
DINO_WEIGHTS = INPUT_FILES["dinov3_vits16_pretrain_lvd1689m-08c60483.pth"]
TRAIN_LAST_BLOCKS = 4
DINO_LR = 1e-5
HEAD_LR = 1e-4
TRIPLET_MARGIN = .3      # unused by SupCon, kept for the loss interface
METRIC_LOSS = "supcon"   # supervised multi-positive InfoNCE / NT-Xent
REID_TEMPERATURE = .1
RESUME_FROM = None       # e.g. Path("/kaggle/input/<previous-output>/latest.pt")

mining_config = dict(enabled=False, warmup_epochs=5, refresh_every=5, top_k=20,
                     hard_fraction=HARD_FRACTION, aggregation="max_crop_cosine")
reid_config = dict(hub_repo=str(DINOV3_DIR), embedding_dim=256, train_last_blocks=TRAIN_LAST_BLOCKS,
                   crop_size=REID_CROP_SIZE, feature_pooling="cls_upper_lower")
reid = DinoV3CropReID(**reid_config, weights_path=DINO_WEIGHTS).to(device)
model = DinoPersonSearch(detector, reid)
run_config = dict(experiment="swin_frozen_dinov3_last4_cls_upper_lower_supcon_p16k4_256x128", seed=SEED,
                  val_fraction=VAL_FRACTION, image_size=IMAGE_SIZE, query_crop_size=REID_CROP_SIZE, P=P, K=K,
                  epochs=NUM_EPOCHS, reid_config=reid_config, detector_config=detector_config,
                  detector_checkpoint=str(DETECTOR_CHECKPOINT), detector_epoch=DETECTOR_EPOCH,
                  dino_frozen=False, dino_lr=DINO_LR, mining_config=mining_config, dino_weights=str(DINO_WEIGHTS),
                  head_lr=HEAD_LR, triplet_margin=TRIPLET_MARGIN, metric_loss=METRIC_LOSS,
                  reid_temperature=REID_TEMPERATURE, gallery_embeddings="standalone_detected_crops",
                  score_threshold=.5, checkpoint_selection="validation_retrieval_mAP", val_retrieval_every=1)
optimizer = torch.optim.AdamW([
    {"params": [p for p in reid.backbone.parameters() if p.requires_grad], "lr": DINO_LR},
    {"params": reid.embedding_proj.parameters(), "lr": HEAD_LR},
], weight_decay=1e-4)
scaler = make_grad_scaler(device)
checkpoint_dir = WORKING / "checkpoints" / "dinov3_reid"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

start_epoch, best_val, best_retrieval_map = 0, float("inf"), -1.
if RESUME_FROM is not None:
    state = torch.load(RESUME_FROM, map_location="cpu", weights_only=False)
    if state["run_config"]["experiment"] != run_config["experiment"] or state["reid_config"] != reid_config:
        raise ValueError("RESUME_FROM is a different experiment")
    best_retrieval_map, best_val, start_epoch = state["best_retrieval_map"], state["best_val"], state["epoch"]
    random.setstate(state["python_rng"])
    torch.random.set_rng_state(state["torch_rng"])
    if torch.cuda.is_available() and state["cuda_rng"] is not None:
        torch.cuda.set_rng_state_all(state["cuda_rng"])
    reid.load_state_dict(state["reid_state_dict"])
    optimizer.load_state_dict(state["optimizer_state_dict"])
    scaler.load_state_dict(state["scaler_state_dict"])
    del state

run = start_run(run_config, name="prw-dinov3-last4-cls-upper-lower-infonce-p16-k4-256x128")
validation_names = [full_train.image_names[i] for i in val_dataset.indices]
validation_cache = build_validation_gallery(model, PRW_ROOT, validation_names, device, score_threshold=.5)
print("validation gallery:", len(validation_names), "frames,", len(validation_cache["queries"]), "queries")

for epoch in range(start_epoch, NUM_EPOCHS):
    train_sampler.set_epoch(epoch)
    crop_epoch(reid, train_loader, device, optimizer, scaler, TRIPLET_MARGIN, run, epoch,
               metric_loss=METRIC_LOSS, temperature=REID_TEMPERATURE)
    metrics = crop_epoch(reid, val_loader, device, margin=TRIPLET_MARGIN, run=run, epoch=epoch,
                         max_batches=VAL_MAX_BATCHES, metric_loss=METRIC_LOSS, temperature=REID_TEMPERATURE)
    retrieval_val = evaluate_validation_gallery(model, PRW_ROOT, validation_cache, device)
    run.log({"epoch": epoch + 1, **{f"val/{k}": v for k, v in retrieval_val.items()}})
    improved = retrieval_val["retrieval_mAP"] > best_retrieval_map
    best_retrieval_map = max(best_retrieval_map, retrieval_val["retrieval_mAP"])
    best_val = min(best_val, metrics["metric_loss"])
    state = dict(epoch=epoch + 1, best_val=best_val, best_retrieval_map=best_retrieval_map,
                 validation_retrieval=retrieval_val, run_config=run_config,
                 reid_config=reid_config, detector_config=detector_config,
                 detector_state_dict=model.detector.state_dict(), reid_state_dict=reid.state_dict(),
                 optimizer_state_dict=optimizer.state_dict(), scaler_state_dict=scaler.state_dict(),
                 mined_neighbors={}, last_mining_epoch=None,
                 python_rng=random.getstate(), torch_rng=torch.random.get_rng_state(),
                 cuda_rng=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None)
    torch.save(state, checkpoint_dir / "latest.pt")
    if improved:
        torch.save(state, checkpoint_dir / "best.pt")
    print(f"epoch {epoch + 1}: val loss={metrics['metric_loss']:.4f}, val mAP={retrieval_val['retrieval_mAP']:.4f}, "
          f"best mAP={best_retrieval_map:.4f}", flush=True)

shutil.copy2(checkpoint_dir / "best.pt", WORKING / "swin_dino_reid.pt")
if USE_WANDB:
    artifact = wandb.Artifact("prw-dinov3-upper-lower-best-model", type="model", metadata=run_config)
    artifact.add_file(str(WORKING / "swin_dino_reid.pt"))
    run.log_artifact(artifact)
run.finish()
print("saved", WORKING / "swin_dino_reid.pt")

## 5. Test evaluation with the official `eval_search_prw`
The best encoder is evaluated once on the official test split. It is paired with the validation detector used during training (256×384), so this number is **not** the one in `main.ipynb`. There the same encoder uses the fine-tuned 512×768 detector, which finds more people.

> **Run time (Kaggle T4):** ≈ 15 min.

In [ ]:
from prw_evaluation import evaluate_prw

best_state = torch.load(checkpoint_dir / "best.pt", map_location="cpu", weights_only=False)
reid.load_state_dict(best_state["reid_state_dict"], strict=True)
print("best epoch:", best_state["epoch"])
del best_state
model.eval()
result = evaluate_prw(model, PRW_ROOT, device, score_threshold=.5, iou_threshold=.5,
                      detector_image_size=IMAGE_SIZE, gallery_embedding_mode="crop", official_eval=True,
                      batch_size=2, query_batch_size=32, gallery_crop_batch_size=32)
print(f"DINOv3 + joint Swin-T detector: official mAP = {result['official_mAP']:.2%}, top-1 = {result['official_top1']:.2%}")